In [1]:
using Pkg 

In [2]:
using JuMP, Gurobi
using Plots

## Tables-and-chairs revisited
Here is the tables-and-chairs example we saw previously, together with the code:

$$
\begin{align*}
\max_{x_1,x_2} \quad
& 20 x_1 + 30 x_2 
\\
\text{such that} \quad
& x_1 + 2 x_2 \leq 30 \\
& 10 x_1 + 5 x_2 \leq 120 \\
& x_1, x_2 \geq 0.
\end{align*}
$$

In [ ]:
model = Model(Gurobi.Optimizer)
@variable(model, x[1:2] >= 0)
@objective(model, Max, 20x[1] + 30x[2])
# Named constraint
@constraint(model, labor, x[1] + 2x[2] <= 30)
@constraint(model, wood, 10x[1] + 5x[2] <= 120)
optimize!(model)

Set parameter Username
Set parameter LicenseID to value 2666310
Academic license - for non-commercial use only - expires 2026-05-14
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M2 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0xdadcd0d8
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [2e+01, 3e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 1e+02]
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.0000000e+31   3.375000e+30   5.000000e+01      0s
       2    4.8000000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  4.800000000e+02

User-callback calls 47, time in user-callback 0.00 sec


We saw last time how to query the optimal solution and optimal objective values:

In [6]:
println("Optimal objective value: ", objective_value(model))
println("Optimal solution: x1 = ", value(x[1]), ", x2 = ", value(x[2]))

Optimal objective value: 480.0
Optimal solution: x1 = 6.0, x2 = 12.0


What's new here is we can also directly query the dual solution from the model. There are various ways to do this:

In [15]:
println("Optimal dual solution for labor constraint: (using shadow_price()): ", shadow_price(labor))
println("Optimal dual solution for wood constraint: (using shadow_price()):  ", shadow_price(wood))

Optimal dual solution for labor constraint: (using shadow_price()): 13.333333333333334
Optimal dual solution for wood constraint: (using shadow_price()):  0.6666666666666665


In [16]:
println("Optimal dual solution for labor constraint: (using dual()): ", dual(labor))
println("Optimal dual solution for wood constraint: (using dual()):  ", dual(wood))

Optimal dual solution for labor constraint: (using dual()): -13.333333333333334
Optimal dual solution for wood constraint: (using dual()):  -0.6666666666666665


Why are there two functions and why do they do different things? This is unique to the JuMP package; they have a `dual` function which works more generally but is independent of the objective sense (Max or Min), and a `shadow_price` function which works specific to linear programming. Read more here: https://jump.dev/JuMP.jl/stable/api/JuMP/#shadow_price

## Exercise 1

Here is the dual of the tables-and-chairs problem:

$$
\begin{align*}
\min_{p_1,p_2} \quad
& 30 p_1 + 120 p_2 
\\
\text{such that} \quad
& p_1 + 10 p_2 \geq 20
\\
& 2 p_1 + 5 p_2 \geq 30 
\\
& p_1, p_2 \geq 0
\end{align*}
$$

(a) Formulate and solve the dual problem, and verify that you get the same dual solution as before.

(b) Plot the feasible region and optimal solution of the dual problem. 

## Exercise 2: Sensitivity analysis

We saw the interpretation of the shadow price of labor and wood: each additional unit of labor and increases profit by $\frac{40}{3}$ and $\frac{2}{3}$ respectively. When does this hold until?

Hint: write the labor-wood LP as a function that takes two parameters: the amount of labor availability and wood availability, and returns the optimal primal solution and dual solution. Then use that function to test different values and form a hypothesis.